# Spain oral-health workforce administrative indicators by territory and nominal year, 2024–2025 Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.s37e-wer3/fair2.json`

This notebook emphasizes using Croissant entity identifiers (`@id`) consistently when referring to record sets, fields, and related metadata.


In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the Croissant metadata from the dataset URL and inspect the dataset-level information.

Important note: `dataset.metadata` is handled as a single metadata object. We convert it to JSON only when we want a plain serializable structure for inspection.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style="whitegrid")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.s37e-wer3/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset loaded successfully")
print(f"Name: {metadata_json.get('name')}")
print(f"Description: {metadata_json.get('description')}")
print(f"Identifier: {metadata_json.get('identifier')}")
print(f"Version: {metadata_json.get('version')}")
print(f"License: {metadata_json.get('license')}")

In [ ]:
# Display a compact dataset-level summary
dataset_summary = {
    "@id": metadata_json.get("@id"),
    "name": metadata_json.get("name"),
    "temporalCoverage": metadata_json.get("temporalCoverage"),
    "spatialCoverage": metadata_json.get("spatialCoverage"),
    "keywords": metadata_json.get("keywords"),
    "dataUseCases": metadata_json.get("dataUseCases"),
}
pprint(dataset_summary)

## 2. Data Overview
Review the metadata structure and identify available Croissant entities by `@id`.

The package metadata included with this dataset indicates that `recordSet` is currently empty at the top-level metadata snapshot. Since Croissant packages can vary in how record-bearing resources are exposed, the code below inspects metadata carefully and then attempts to list any record sets made available through the loaded Croissant object.

All references below use `@id` wherever available.

In [ ]:
# Inspect top-level metadata keys
print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

record_sets_from_json = metadata_json.get("recordSet", [])
print(f"\nNumber of record sets declared in metadata JSON: {len(record_sets_from_json)}")
if record_sets_from_json:
    for rs in record_sets_from_json:
        print(rs.get("@id"), rs.get("name"))
else:
    print("No record sets are explicitly listed in the provided metadata JSON snapshot.")

In [ ]:
# Try to inspect the dataset object for record set metadata in a library-agnostic way
available_record_set_ids = []

if hasattr(metadata_obj, "record_sets"):
    try:
        for rs in metadata_obj.record_sets:
            rs_json = rs.to_json() if hasattr(rs, "to_json") else rs
            rs_id = rs_json.get("@id") if isinstance(rs_json, dict) else None
            if rs_id:
                available_record_set_ids.append(rs_id)
        print("Record sets discovered via metadata_obj.record_sets:")
        print(available_record_set_ids)
    except Exception as e:
        print("Could not inspect metadata_obj.record_sets:", e)
else:
    print("metadata_obj does not expose a `record_sets` attribute in this environment.")

if not available_record_set_ids and record_sets_from_json:
    available_record_set_ids = [rs.get("@id") for rs in record_sets_from_json if isinstance(rs, dict) and rs.get("@id")]

print("\nFinal discovered record set IDs:")
print(available_record_set_ids)

### Example: inspecting records from a record set by `@id`
The template notebook includes iterating through `dataset.records(record_set=<record_set_id>)`. Since this dataset may not expose record sets directly in the provided metadata snapshot, the next cell demonstrates the pattern safely.

If one or more record set `@id`s are discovered, the code prints a few sample records from the first one.

In [ ]:
if available_record_set_ids:
    sample_record_set_id = available_record_set_ids[0]
    print(f"Sampling records from record set @id: {sample_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record set @id was discovered, so record iteration is skipped.")

## 3. Data Extraction
Load records from each discovered record set into pandas DataFrames.

This section uses variables so the notebook adapts dynamically to the dataset structure. If no record sets are available through the Croissant interface, the notebook still provides a metadata-driven exploration path.

In [ ]:
# Extract data from each discovered record set
record_set_ids = available_record_set_ids
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if dataframes:
    first_record_set_id = next(iter(dataframes))
    print(f"\nColumns for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No DataFrames were created because no record sets could be loaded from the dataset.")

### Metadata-derived helper tables
Even if record-level extraction is unavailable, the dataset metadata itself contains useful analytical information. Below we convert several dataset-level arrays into tidy DataFrames for exploration.

In [ ]:
# Build metadata-driven tables for exploration
list_fields = [
    "keywords",
    "dataBiases",
    "dataLimitations",
    "dataUseCases",
    "dataPreprocessingProtocol",
    "personalSensitiveInformation",
    "dataCollectionTimeframe",
    "dataCollectionType",
]

metadata_tables = {}
for field_id in list_fields:
    values = metadata_json.get(field_id, [])
    if isinstance(values, list):
        metadata_tables[field_id] = pd.DataFrame({field_id: values})
    else:
        metadata_tables[field_id] = pd.DataFrame({field_id: [values]})

for field_id, df in metadata_tables.items():
    print(f"{field_id}: {df.shape[0]} rows")

display(metadata_tables["keywords"])
display(metadata_tables["dataUseCases"])

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps to whichever tabular structures are available.

Because this dataset snapshot may not expose record sets directly, this section includes:

1. **Record-level EDA** when tabular record sets are available.
2. **Metadata-level EDA** using fields such as `dataBiases`, `dataLimitations`, and `dataUseCases`.

This keeps the notebook robust while still following the template's exploratory workflow.

In [ ]:
# Attempt record-level EDA dynamically if any numeric fields exist
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Working record set @id: {record_set_id}")
    print(f"Numeric columns discovered: {numeric_columns}")
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        std = filtered_df[numeric_field_id].std()
        if pd.notna(std) and std != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / std
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0.0
        
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object']
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=True).reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for record-level grouping.")
    else:
        print("No numeric fields found for record-level EDA.")
else:
    print("No record-level DataFrames available. Proceeding with metadata-level EDA only.")

In [ ]:
# Metadata-level EDA: text length analysis for descriptive list fields
text_analysis_rows = []
for field_id, df_meta in metadata_tables.items():
    for value in df_meta[field_id].dropna().astype(str):
        text_analysis_rows.append({
            "field_@id": field_id,
            "text": value,
            "char_length": len(value),
            "word_count": len(value.split())
        })

text_analysis_df = pd.DataFrame(text_analysis_rows)
display(text_analysis_df.head())

field_summary_df = (
    text_analysis_df.groupby("field_@id")
    .agg(entries=("text", "count"), avg_char_length=("char_length", "mean"), avg_word_count=("word_count", "mean"))
    .sort_values("entries", ascending=False)
    .reset_index()
)
display(field_summary_df)

## 5. Visualization
Visualize the available data.

We create:

- a bar chart summarizing counts of entries across metadata fields,
- a histogram of metadata text lengths,
- and, if record-level numeric data are available, a histogram for a numeric field.

In [ ]:
# Visualization 1: number of entries per metadata field
plt.figure(figsize=(10, 5))
sns.barplot(data=field_summary_df, x="field_@id", y="entries", palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Number of entries by metadata field @id")
plt.xlabel("Metadata field @id")
plt.ylabel("Count of entries")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: distribution of text lengths in metadata list fields
plt.figure(figsize=(8, 5))
sns.histplot(text_analysis_df["word_count"], bins=15, kde=True, color="steelblue")
plt.title("Distribution of metadata text lengths (word count)")
plt.xlabel("Word count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: record-level numeric distribution, if available
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color="darkorange")
        plt.title(f"Distribution of numeric field: {numeric_field_id}\n(record set @id: {record_set_id})")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric record-level fields available for plotting.")
else:
    print("No record-level DataFrames available for numeric visualization.")

## 6. Conclusion
This notebook demonstrated how to:

- load a Croissant dataset from a schema URL using `mlcroissant`,
- inspect dataset-level metadata safely,
- discover record sets dynamically by `@id` where available,
- extract records into pandas DataFrames,
- and perform both record-level and metadata-level exploratory analysis.

For this dataset, the provided metadata snapshot contains rich descriptive information about collection methods, limitations, biases, and use cases. If record sets are exposed through the Croissant endpoint in your execution environment, the same notebook will automatically load and analyze them using their discovered `@id` values.

A careful reading of the metadata shows that this dataset should be used primarily for descriptive, provenance-aware analysis rather than direct national aggregation or strong temporal comparisons.